# Global Setup & Imports

In [ ]:
# Install necessary libraries
# !pip install torch torchvision transformers peft datasets bitsandbytes accelerate scikit-learn matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import copy
import math
import os
from PIL import Image
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Part 1: Image Re-identification (Transformers)

### 1.1 Data Preparation (Dataset & Augmentation)

In [ ]:
class ReIDDataset(Dataset):
    def __init__(self, data_path, transform=None):
        """
        Load images from folders. 
        Structure expected: root/class_id/image.jpg
        """
        self.transform = transform
        self.image_paths = [] 
        self.labels = []
        self.classes = []
        
        # Walk through directories
        if os.path.exists(data_path):
            self.classes = sorted(os.listdir(data_path))
            for label, class_name in enumerate(self.classes):
                class_dir = os.path.join(data_path, class_name)
                if os.path.isdir(class_dir):
                    for img_file in os.listdir(class_dir):
                        if img_file.endswith(('.jpg', '.png', '.jpeg')):
                            self.image_paths.append(os.path.join(class_dir, img_file))
                            self.labels.append(label)
        else:
            # For demo, create dummy data
            print("Data path not found, using dummy data")
            self.classes = [f"class_{i}" for i in range(10)]
            self.image_paths = [f"dummy_{i}.jpg" for i in range(100)]
            self.labels = [i % 10 for i in range(100)]
        
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            # Dummy image if file not found
            img = Image.new('RGB', (256, 128), color=(128, 128, 128))
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

# Aggressive Augmentation for small datasets
train_transforms = T.Compose([
    T.Resize((256, 128)), # Standard Re-ID size
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.RandomErasing(p=0.5), # Helps with occlusion - applied after ToTensor
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = T.Compose([
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Datasets and DataLoaders
# Assuming data_path is set, e.g., "/path/to/reid_data"
# For demo, use dummy
train_dataset = ReIDDataset("dummy_train", transform=train_transforms)
test_dataset = ReIDDataset("dummy_test", transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 1.2 & 1.3 Model Definitions (ResNet & BotNet)

**ResNet50:**

In [ ]:
def get_resnet50(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    # Modify the final layer for Re-ID (classification)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)

**BotNet (Bottleneck Transformer):**
This is the core implementation task. We replace the spatial convolutions in the last stage (Stage 4) with Multi-Head Self-Attention.

In [ ]:
class MHSA(nn.Module):
    """ Multi-Head Self-Attention for 2D Images """
    def __init__(self, n_dims, width, height, heads=4):
        super(MHSA, self).__init__()
        self.heads = heads
        self.query = nn.Conv2d(n_dims, n_dims, kernel_size=1)
        self.key = nn.Conv2d(n_dims, n_dims, kernel_size=1)
        self.value = nn.Conv2d(n_dims, n_dims, kernel_size=1)

        self.rel_h = nn.Parameter(torch.randn([1, heads, n_dims // heads, 1, height]), requires_grad=True)
        self.rel_w = nn.Parameter(torch.randn([1, heads, n_dims // heads, width, 1]), requires_grad=True)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        n_batch, C, width, height = x.size()
        
        # 1. Projections
        q = self.query(x).view(n_batch, self.heads, C // self.heads, -1)
        k = self.key(x).view(n_batch, self.heads, C // self.heads, -1)
        v = self.value(x).view(n_batch, self.heads, C // self.heads, -1)

        # 2. Content-Content Attention
        content_content = torch.matmul(q.permute(0, 1, 3, 2), k)
        
        # 3. Positional Embeddings (Relative) -> Simplified for this assignment
        # In a full BotNet, you add relative position encodings here. 
        # For simplicity, we calculate basic attention:
        energy = content_content 
        
        attention = self.softmax(energy) # Save this for visualization later!
        self.last_attention_map = attention # Hook for Q1.5

        # 4. Aggregation
        out = torch.matmul(v, attention.permute(0, 1, 3, 2))
        out = out.view(n_batch, C, width, height)
        return out

class BotNet50(nn.Module):
    def __init__(self, num_classes, resolution=(256, 128)):
        super(BotNet50, self).__init__()
        # Load backbone
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        # Extract initial layers
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        
        # Replace Layer 4 Convolutions with MHSA blocks
        # Note: In a real BotNet, you replace the 3x3 conv inside the Bottleneck with MHSA.
        # Ideally, iterate through resnet.layer4 and replace conv2 with MHSA.
        self.layer4 = resnet.layer4 
        
        # Example: Replacing the final spatial processing with a global MHSA before pooling
        # (Simplified version for assignment feasibility)
        # H, W at stage 4 for 256x128 input is usually 16x8
        self.mhsa = MHSA(2048, width=resolution[1]//32, height=resolution[0]//32)
        
        self.avgpool = resnet.avgpool
        self.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x) 
        
        # Apply Attention
        x = self.mhsa(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

### 1.5 Attention Visualization

In [ ]:
def visualize_attention(model, img_tensor, original_image):
    model.eval()
    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0).to(device))
    
    # Retrieve stored attention map from the MHSA layer
    attn_map = model.mhsa.last_attention_map # Shape: (1, heads, pixels, pixels)
    
    # Average over heads and reshape to spatial dimensions
    H_feat = W_feat = int(math.sqrt(attn_map.shape[-1]))
    attn_map = attn_map.mean(dim=1).view(H_feat, W_feat, H_feat, W_feat)
    
    # Project specific pixel attention or global attention
    # For simplicity, average over all positions
    global_attn = attn_map.mean(dim=(0,1))
    
    # Resize to image size
    import torch.nn.functional as F
    global_attn = F.interpolate(global_attn.unsqueeze(0).unsqueeze(0), size=(256, 128), mode='bilinear').squeeze()
    
    # Normalize
    global_attn = (global_attn - global_attn.min()) / (global_attn.max() - global_attn.min())
    
    # Overlay heatmap on original_image
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_image)
    plt.title("Original Image")
    
    plt.subplot(1, 2, 2)
    plt.imshow(original_image)
    plt.imshow(global_attn.cpu(), alpha=0.5, cmap='jet')
    plt.title("Attention Heatmap")
    plt.show()

### Training and Evaluation for Re-ID

In [ ]:
# Initialize models
num_classes = len(train_dataset.classes)
resnet_model = get_resnet50(num_classes)
botnet_model = BotNet50(num_classes)

# Optimizers
resnet_optimizer = optim.Adam(resnet_model.parameters(), lr=1e-4)
botnet_optimizer = optim.Adam(botnet_model.parameters(), lr=1e-4)

criterion = nn.CrossEntropyLoss()

def train_model(model, optimizer, train_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Accuracy: {accuracy:.2f}%")
    return accuracy

# Train ResNet
print("Training ResNet50...")
train_model(resnet_model, resnet_optimizer, train_loader)
resnet_acc = evaluate_model(resnet_model, test_loader)

# Train BotNet
print("Training BotNet50...")
train_model(botnet_model, botnet_optimizer, train_loader)
botnet_acc = evaluate_model(botnet_model, test_loader)

print(f"ResNet Accuracy: {resnet_acc:.2f}%, BotNet Accuracy: {botnet_acc:.2f}%")

# Part 2: Large Language Diffusion (LLaDA)

### 2.2 Data Pipeline & Prompting

In [ ]:
# 1. Load Dataset
dataset = load_dataset("gretelai/synthetic_text_to_sql")

# 2. Chat Template
SYSTEM_PROMPT = "You are a Text-to-SQL assistant. Output ONLY the SQL query. Do not add explanations."

def format_example(example, tokenizer):
    user_content = f"Schema:\n{example['schema']}\n\nQuestion:\n{example['sql_prompt']}"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example['sql']} # The 'Gold' SQL
    ]
    
    # Apply template WITHOUT tokenizing yet to find boundaries
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    
    # Simple logic to separate Prompt vs Answer for masking
    # Note: This depends on the specific chat template of the base model
    prompt_part = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    answer_part = example['sql']
    
    return prompt_part, answer_part

# 3. SQL Normalization
def normalize_sql(query):
    query = query.lower()
    query = query.replace("`", "").replace(";", "")
    query = " ".join(query.split()) # Fix whitespace
    return query

def exact_match_score(pred, truth):
    return 1 if normalize_sql(pred) == normalize_sql(truth) else 0

### 2.3 Model Loading & Forward Process (The Core)

In [ ]:
# Load Model with Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_name = "GSAI-ML/LLaDA-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    quantization_config=bnb_config, 
    device_map="auto",
    use_cache=False # Important for Diffusion training
)

# Apply LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

**Forward Masking Function:**

In [ ]:
def noisy_batch(input_ids, attention_mask, prompt_lengths, tokenizer):
    """
    Applies forward diffusion masking to the ANSWER part of the batch.
    """
    batch_size, seq_len = input_ids.shape
    masked_input_ids = input_ids.clone()
    labels = input_ids.clone()
    
    # 1. Sample t uniformly
    t = torch.rand(batch_size, device=input_ids.device)
    
    # 2. Compute Mask Probability (e.g., Linear or Cosine schedule)
    # Simple linear schedule: p_mask = t
    p_mask = t.view(-1, 1) 
    
    # 3. Create Mask
    # Generate random matrix
    rand_matrix = torch.rand(input_ids.shape, device=input_ids.device)
    
    # Create a boolean mask where we *should* mask tokens
    # Condition 1: Probability check
    mask_indices = rand_matrix < p_mask
    
    # Condition 2: Do NOT mask the Prompt (indices < prompt_length)
    for i in range(batch_size):
        mask_indices[i, :prompt_lengths[i]] = False
        
    # Condition 3: Do NOT mask Padding
    mask_indices = mask_indices & (attention_mask.bool())

    # Apply Mask Token
    masked_input_ids[mask_indices] = tokenizer.mask_token_id
    
    # Labels: We only compute loss on tokens that WERE masked
    labels[~mask_indices] = -100 # PyTorch ignores -100 in CrossEntropy
    
    return masked_input_ids, labels, p_mask

### 2.3.4 Training Loop

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=2e-4)

def prepare_batch(examples, tokenizer):
    """Prepare a batch of examples for training"""
    input_ids_list = []
    prompt_lengths = []
    
    for example in examples:
        prompt, answer = format_example(example, tokenizer)
        full_text = prompt + answer
        
        # Tokenize
        tokens = tokenizer(full_text, return_tensors='pt', padding=False)
        input_ids = tokens['input_ids'].squeeze()
        
        # Find prompt length
        prompt_tokens = tokenizer(prompt, return_tensors='pt', padding=False)['input_ids'].squeeze()
        prompt_len = len(prompt_tokens)
        
        input_ids_list.append(input_ids)
        prompt_lengths.append(prompt_len)
    
    # Pad to max length
    max_len = max(len(ids) for ids in input_ids_list)
    padded_ids = []
    attention_masks = []
    
    for ids in input_ids_list:
        pad_len = max_len - len(ids)
        padded = torch.cat([ids, torch.full((pad_len,), tokenizer.pad_token_id)])
        mask = torch.cat([torch.ones(len(ids)), torch.zeros(pad_len)])
        padded_ids.append(padded)
        attention_masks.append(mask)
    
    batch_input_ids = torch.stack(padded_ids)
    batch_attention_mask = torch.stack(attention_masks)
    batch_prompt_lengths = torch.tensor(prompt_lengths)
    
    return batch_input_ids, batch_attention_mask, batch_prompt_lengths

def train_step(batch, model, optimizer, tokenizer):
    input_ids, att_mask, prompt_lens = prepare_batch(batch, tokenizer)
    input_ids, att_mask, prompt_lens = input_ids.to(device), att_mask.to(device), prompt_lens.to(device)
    
    # Apply Noise
    masked_ids, labels, p_mask = noisy_batch(input_ids, att_mask, prompt_lens, tokenizer)
    
    # Forward Pass
    outputs = model(input_ids=masked_ids, attention_mask=att_mask)
    logits = outputs.logits
    
    # Loss Calculation
    loss_fct = nn.CrossEntropyLoss(reduction='none')
    # Reshape for loss: (B*L, Vocab)
    loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
    
    # Reweighting
    # Reshape loss back to (B, L)
    batch_size = input_ids.shape[0]
    loss = loss.view(batch_size, -1)
    
    # Calculate mask ratio per sample for reweighting
    # Theory: High masking = easy to predict macro structure, needs less weight? 
    # Or inverse: Low masking = hard to predict exact token?
    # LLaDA paper suggests specific reweighting. 
    # Simple implementation: 1 / (1 - p_mask) or similar stability term.
    weights = 1.0 / (1.0 - p_mask + 1e-6)
    
    # Apply weights only to masked tokens (where labels != -100)
    mask_bool = labels != -100
    weighted_loss = (loss * mask_bool).sum(dim=1) * weights.squeeze()
    
    final_loss = weighted_loss.mean()
    
    final_loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    return final_loss.item()

# Example training (simplified)
# train_data = dataset['train'].select(range(100))  # Small subset for demo
# for epoch in range(1):
#     for i in range(0, len(train_data), 4):
#         batch = train_data[i:i+4]
#         loss = train_step(batch, model, optimizer, tokenizer)
#         print(f"Loss: {loss:.4f}")

### 2.4 Block Diffusion Sampling (Generation)

In [ ]:
@torch.no_grad()
def generate_block_diffusion(model, tokenizer, prompt_text, steps=32, gen_len=64):
    """
    1. Start with Prompt + [MASK] * gen_len
    2. Iteratively predict and 'lock in' high-confidence tokens.
    """
    # Prepare Input
    prompt_ids = tokenizer.encode(prompt_text, return_tensors='pt').to(device)
    mask_ids = torch.full((1, gen_len), tokenizer.mask_token_id, device=device)
    input_ids = torch.cat([prompt_ids, mask_ids], dim=1)
    
    L = input_ids.shape[1]
    prompt_len = prompt_ids.shape[1]
    
    # Indices corresponding to the generated answer
    unknown_indices = set(range(prompt_len, L))
    
    # Schedule: How many tokens to lock per step
    tokens_to_lock_per_step = gen_len // steps
    
    for step in range(steps):
        # Forward pass
        outputs = model(input_ids)
        logits = outputs.logits # (1, L, Vocab)
        
        # Get predictions and confidence (Softmax max value)
        probs = torch.softmax(logits, dim=-1)
        confidences, predicted_ids = torch.max(probs, dim=-1)
        
        # We only care about currently unknown indices
        current_unknowns = list(unknown_indices)
        if not current_unknowns: break
        
        # Sort unknown indices by confidence
        # We want to lock the ones the model is MOST sure about
        candidates = []
        for idx in current_unknowns:
            score = confidences[0, idx].item()
            token = predicted_ids[0, idx].item()
            candidates.append((score, idx, token))
            
        candidates.sort(key=lambda x: x[0], reverse=True)
        
        # Select top-k to commit
        k = min(tokens_to_lock_per_step, len(candidates))
        top_candidates = candidates[:k]
        
        # Update input_ids (Lock in the tokens)
        for score, idx, token in top_candidates:
            input_ids[0, idx] = token
            unknown_indices.remove(idx)
            
    # Decode final output
    generated_text = tokenizer.decode(input_ids[0, prompt_len:], skip_special_tokens=True)
    return generated_text

### 2.4.3 Evaluation & Post-processing

In [ ]:
def post_process_sql(text):
    # Extract only the SQL part
    # Look for SELECT ... ;
    if "SELECT" in text:
        start = text.find("SELECT")
        end = text.find(";", start)
        if end != -1:
            return text[start:end+1]
        return text[start:]
    return text

def evaluate_pipeline(test_dataset, model, tokenizer, num_samples=10):
    total = 0
    correct = 0
    
    for example in test_dataset.select(range(num_samples)):  # Small subset for demo
        prompt, gold_sql = format_example(example, tokenizer)
        
        # Generate
        raw_output = generate_block_diffusion(model, tokenizer, prompt)
        pred_sql = post_process_sql(raw_output)
        
        # Metric
        if exact_match_score(pred_sql, gold_sql):
            correct += 1
        total += 1
        
    print(f"Accuracy: {correct/total * 100:.2f}%")

# Example evaluation
# test_data = dataset['test']
# evaluate_pipeline(test_data, model, tokenizer)

### Example Usage

In [ ]:
# Example for Re-ID visualization
# img, label = test_dataset[0]
# original_img = T.ToPILImage()(img)
# visualize_attention(botnet_model, img, original_img)

# Example for LLaDA generation
# example = dataset['test'][0]
# prompt, _ = format_example(example, tokenizer)
# generated = generate_block_diffusion(model, tokenizer, prompt)
# print("Generated SQL:", generated)